# Observable - Xarray implementation example

This notebook aims at demonstrating the specificities of the `Observable` class implementation based on `xarray.Dataset`.

In [ ]:
# Setup
from helpers import make_file

from acm import setup_logging

setup_logging()

data = make_file("observable.h5", backend="xarray")

The `XarrayObservable` class handles observables stored in a `xarray.Dataset` object. It is a subclass of `BaseObservable` and implements the same interface, so it can be used interchangeably with other observable types.

The data objects (`x`, `y`, ...) are stored as `xarray.DataArray` objects in the dataset, which provide a convenient interface for working with labeled multi-dimensional arrays.

### Object requirements
The loaded object requires at least `x` and `y` DataArrays.

To allow reshaping on 2-dimensional arrays, each of those data objects also requires a `sample` and `features` attribute, which are lists of dimension names used to identify the dimensions to flatten on for each axis. Due to the Dataset structure, some data objects might be reshaped on extra axis with `NaN` values (see [xarray missing values](https://docs.xarray.dev/en/latest/get-help/faq.html#how-does-xarray-handle-missing-values)). If the dimensions containing them are known beforehand, the data object can also contain a `nan_dims` attribute, which is a list of dimension names that should drop the `NaN` values when reshaping.

### Choice of backend
**Pros**
- Array properties: The `xarray.DataArray` type inherits from `numpy.ndarray`, which allows for efficient array operations and broadcasting. It also provides additional functionality, such as labeled dimensions and coordinates, which can be useful for working with multi-dimensional data.
- Native filtering: Filters are directly applied on the dataset, which inherits from numpy and pandas selection methods. This makes filtering and selection fast and efficient with no extra internal steps.
- Accessible data properties: The `getattr` method exposes the entire (filtered) dataset API - useful to explore the data and its properties. Also, the API is very similar to that of numpy and pandas, making it easy to work with.

**Cons**
- Strict indexing structure:
  - Sparse indexing not allowed: every sample needs to share an identical nested structure (e.g. the same number of HOD per cosmology), or otherwise reindexed - loosing information.
  - Flattening structure needs to be known beforehand: the `sample` and `features` attributes must be defined for each data object.
  - Combination of DataArrays with different indexes on same dimensions fills unknown indexes with `NaN` values, mitigated by the `nan_dims` attribute when known - not always possible to know beforehand.
- Prediction filtering is somewhat expensive: The prediction array needs to be cast on a reshaped DataArray, filtered then recast to numpy.

In [ ]:
from acm.observables.xarray import XarrayObservable

obs = XarrayObservable(data=data)
obs

> Directly passing data to the `XarrayObservable` class allows missing data elements (no checks are performed on the data structure). 
> This is done on purpose to allow custom data structures for specific cases (e.g. measurements with no `x` value, ...)

In [ ]:
# Accessing raw DataArray objects trough raw=True
raw_y = obs.get_data("y", raw=True)
type(raw_y), raw_y.dims, raw_y.coords

> Note: `xarray` allows slice selection on any coordinate, with rebinning

In [ ]:
# nested=True preserves the original (filtered but unflattened) DataArray shape
obs.clear_filters()
obs.set_filters(ell=[0, 2])

s1 = obs.get_data("y").shape          # flattened 2D: (n_samples, n_ell * n_k)
s2 = obs.get_data("y", nested=True).shape   # unflattened: (n_i, n_j, n_ell, n_k)

s1, s2

The class also expose several methods to access some specific properties from the filtered dataset.

`get_coordinate_list`: returns the list of coordinates for a given dimension, after applying the filters.

In [ ]:
# Coordinate lists, read from the (filtered) dataset's coordinates
obs.set_filters(i=[0], ell=[0, 2])
ells = obs.get_coordinate_list("ell")
k = obs.get_coordinate_list("k")

ells, k

The class exposes `__getattr__` to allow the direct access to the underlying `xarray.Dataset` attributes (with filters applied), so that the user can use the `xarray` API directly on the observable object. For example, `obs.dims` will return the dimensions of the underlying dataset.

In [ ]:
# Anything that is not a data variable of the dataset will be accessed trough the getattr method of the dataset.
obs.dims, obs.sizes # dims/sizes as they stand after filtering

In [ ]:
# Data variable names (x, y, covariance_y, ...) short-circuit straight to get_data
obs.y.shape  # equivalent to obs.get_data("y").shape

`get_test_set` is a shortcut method to access the `x_test` and `y_test` data variables (if they exist) as 2D arrays with filters and selection applied.

In [ ]:
x_test_raw = obs.get_data("x_test", raw=True)
x_test_raw.attrs.get("nan_dims") # Those dims are dropped when formatting the data

In [ ]:
x_test, y_test = obs.get_test_set()
x_test.shape, y_test.shape

> Older Observable files can have been saved as pickled numpy objects containing a dictionary (see `acm.utils.xarray`). The `XarrayObservable` class can load those files for backward compatibility, but it is recommended to save new compressed files as `.h5` objects.